In [8]:
import os
import json
import logging
from typing import Optional
from contextlib import asynccontextmanager

In [2]:
from fastapi import FastAPI, HTTPException
from influxdb import InfluxDBClient
import paho.mqtt.client as mqtt


In [3]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("backend")

In [9]:
# Config from env
MQTT_HOST = os.getenv("MQTT_HOST", "mosquitto")
MQTT_PORT = int(os.getenv("MQTT_PORT", 1883))
INFLUX_HOST = os.getenv("INFLUXDB_HOST", "influxdb")
INFLUX_PORT = int(os.getenv("INFLUXDB_PORT", 8086))
INFLUX_DB = os.getenv("INFLUXDB_DB", "ems")
INFLUX_USER = os.getenv("INFLUXDB_USER", "admin")
INFLUX_PASSWORD = os.getenv("INFLUXDB_PASSWORD", "admin123")

influx_client: Optional[InfluxDBClient] = None
mqtt_client: Optional[mqtt.Client] = None

In [10]:
def to_float_safe(v):
    try:
        return float(v)
    except Exception:
        return None

def on_connect(client, userdata, flags, rc):
    logger.info("Connected to MQTT broker with rc=%s", rc)
    # subscribe to all telemetry
    client.subscribe("ems/+/+/telemetry")

def on_message(client, userdata, msg):
    global influx_client
    try:
        payload = msg.payload.decode()
        data = json.loads(payload)
    except Exception as e:
        logger.exception("Malformed MQTT payload: %s", e)
        return

    # expect payload to have device_id and type keys
    device_id = data.get("device_id", "unknown")
    dev_type = data.get("type", "unknown")
    timestamp = data.get("ts")  # optional ISO timestamp
    # build fields from numeric keys
    fields = {}
    for k, v in data.items():
        if k in ("device_id", "type", "ts"):
            continue
        num = to_float_safe(v)
        if num is not None:
            fields[k] = num

    if not fields:
        logger.debug("No numeric fields to write for %s", device_id)
        return

    point = {
        "measurement": "telemetry",
        "tags": {"device_id": device_id, "type": dev_type},
        "fields": fields,
    }
    if timestamp:
        point["time"] = timestamp

    try:
        influx_client.write_points([point])
        logger.debug("Wrote point for %s: %s", device_id, fields)
    except Exception as e:
        logger.exception("Failed to write to InfluxDB: %s", e)


In [11]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    global influx_client, mqtt_client
    # STARTUP
    influx_client = InfluxDBClient(
        host=INFLUX_HOST,
        port=INFLUX_PORT,
        username=INFLUX_USER,
        password=INFLUX_PASSWORD,
        database=INFLUX_DB,
    )
    try:
        influx_client.create_database(INFLUX_DB)
    except Exception:
        pass
    mqtt_client = mqtt.Client()
    mqtt_client.on_connect = on_connect
    mqtt_client.on_message = on_message
    mqtt_client.connect(MQTT_HOST, MQTT_PORT)
    mqtt_client.loop_start()
    logger.info("Backend started: MQTT -> InfluxDB ingestion active")
    yield
    # SHUTDOWN
    if mqtt_client:
        mqtt_client.loop_stop()
        mqtt_client.disconnect()

app = FastAPI(title="EMS Backend (ingestion)", lifespan=lifespan)

@app.get("/")
def root():
    return {"status": "ok", "note": "EMS ingestion backend running"}

@app.get("/metrics/latest")
def metrics_latest(device_id: str):
    """Return the latest telemetry point for device_id (if any)."""
    global influx_client
    if not device_id:
        raise HTTPException(status_code=400, detail="device_id required")
    query = f'SELECT * FROM telemetry WHERE "device_id" = \'{device_id}\' ORDER BY time DESC LIMIT 1'
    try:
        result = influx_client.query(query)
        points = list(result.get_points(measurement="telemetry"))
        if not points:
            return {"device_id": device_id, "found": False}
        return {"device_id": device_id, "found": True, "point": points[0]}
    except Exception as e:
        logger.exception("Query failed: %s", e)
        raise HTTPException(status_code=500, detail="Query failed")